we are gonna build a linear regression model that is going to track how much weight you would gain based on the calories you take as intake

But instead of using the mdoel from sklearn, we are gonna build the model from scratch on our own with proper mathematical formulas and intuitions

In [58]:
import numpy as np

# Linear Regression — Weight Gain from Calories
### Notes for quick reference

---

## The Model
```
ŷ = w * x + b
```
- `x` → calories (input)
- `ŷ` → predicted weight gain (output)
- `w` → slope (how much weight changes per calorie)
- `b` → bias (baseline when calories = 0)
- Both start at `0.0` and are learned from data

---

## Step 1 — Normalise X
```python
X = (X - X.mean()) / X.std()
```
Calories are large (1500–4000). Weight is small (-1 to +3).
Without this, gradients explode and loss goes to `inf`.
Normalising brings X to mean≈0, std≈1 → stable training.
**Always use training mean/std when predicting new values.**

---

## Step 2 — Forward Pass (Predict)
```python
y_pred = w * X + b
```
With current `w` and `b`, compute predictions.
At start (w=0, b=0) → all predictions are 0 → completely wrong.
Gets better every epoch.

---

## Step 3 — Loss (MSE)
```python
loss = (1/N) * np.sum((y - y_pred) ** 2)
```
Measures how wrong the model is right now.
- Squaring → negatives don't cancel, big errors hurt more
- Goal of training → make this number as small as possible

---

## Step 4 — Gradients
```python
dw = (-2/N) * np.sum(X * (y - y_pred))
db = (-2/N) * np.sum(y - y_pred)
```
Derivative of loss w.r.t `w` and `b`.
Tells us which direction increases the loss → we go opposite.
- `dw` large → w is far from optimal → take big step
- `dw` small → w is near optimal → take tiny step
- `dw` = 0 → w is at minimum → stop moving ✓

---

## Step 5 — Update w and b
```python
w -= lr * dw
b -= lr * db
```
Step in the opposite direction of the gradient (downhill).
This is the one line where actual learning happens.

---

## Why dw hits 0 automatically
Linear regression loss is a **perfect convex parabola** — one global minimum, no tricks.
As predictions improve → errors shrink → `dw` shrinks → at the bottom `dw = 0` → `w` stops moving.
**No condition needed. The math guarantees it.**

---

## What are Epochs?
One epoch = one full pass over all training data.
Each pass slightly improves `w` and `b`.
Many small steps > one big step (avoids overshooting the minimum).

```
Epoch 100 | Loss: 0.0158   ← still learning
Epoch 300 | Loss: 0.0154   ← converged
Epoch 1000 | Loss: 0.0154  ← no change, done at 300
```
When loss stops changing → model has converged → extra epochs are wasted compute.

---

## Why set Learning Rate manually?
Controls the step size on the loss curve.

| Learning Rate | What happens |
|---|---|
| Too high (e.g. 1.0) | Overshoots minimum, loss explodes |
| Too low (e.g. 0.000001) | Converges too slow, needs 100k epochs |
| Just right (e.g. 0.01) | Smooth descent, converges in ~300 epochs |

No model can figure out lr itself — to know the right step size you'd need to already know where the minimum is.
`0.01` works well for normalised data. Tune by watching the loss curve.

---

## Evaluation Metrics

| Metric | Formula | Means |
|---|---|---|
| R² | `1 - (ss_res / ss_tot)` | % of variance explained. 1.0 = perfect |
| MAE | `mean(abs(y - ŷ))` | Avg error in kg. All errors treated equally |
| MSE | `mean((y - ŷ)²)` | Avg squared error. Big errors hurt more |
| RMSE | `sqrt(MSE)` | MSE back in kg units. Most interpretable |

> If RMSE >> MAE → you have a few large errors pulling things up.
> If they're close → errors are small and consistent. ✓

---

## Full Flow (one line summary)
```
raw calories → normalise → predict → measure loss →
gradients → update w,b → repeat → evaluate
```

In [59]:
class LinearRegression:
  def __init__(self,learning_rate=0.01,epochs=1000):
    self.lr=learning_rate
    self.epochs=epochs
    self.w=0.0
    self.b=0.0
  def fit(self,X,y):
    X=np.array(X,dtype=float)
    y=np.array(y,dtype=float)
    N=len(X)

    self.x_mean=X.mean()
    self.x_std=X.std()
    X=(X-self.x_mean)/self.x_std
    for epoch in range(self.epochs):
      y_pred=self.w*X+self.b
      loss=(1/N)*np.sum((y-y_pred)**2)
      dw=(-2/N)*np.sum(X*(y-y_pred))
      db=(-2/N)*np.sum(y-y_pred)

      self.w-=self.lr*dw
      self.b-=self.lr*db
      if(epoch+1)%100==0:
        print(f"Epoch {epoch+1} | Loss {loss:.4f} | w: {self.w:.4f} | b: {self.b:.4f}")
  def predict(self,X):
    X=(np.array(X,dtype=float)-self.x_mean)/self.x_std
    return self.w*X+self.b
  def r2_score(self,X,y):
    y=np.array(y,dtype=float)
    y_pred=self.predict(X)
    ss_res=np.sum((y-y_pred)**2)
    ss_tot=np.sum((y-y.mean())**2)
    return 1-(ss_res/ss_tot)
  def mae(self,X,y):
    y=np.array(y,dtype=float)
    return np.mean(np.abs(y-self.predict(X)))
  def mse(self,X,y):
    y=np.array(y,dtype=float)
    return np.mean((y-self.predict(X))**2)
  def rmse(self,X,y):
    return np.sqrt(self.mse(X,y))
  def evaluate(self,X,y):
    print(f"R^2 : {self.r2_score(X,y):.4f} (1.0) = perfect")
    print(f"MAE : {self.mae(X,y):.4f} kg (avg absolute error)")
    print(f"MSE : {self.mse(X,y):.4f} kg^2 (penalises big errors)")
    print(f"RMSE : {self.rmse(X,y):.4f} kg (error in original units)")

In [60]:
model=LinearRegression(learning_rate=0.01,epochs=1000)

In [61]:
from sklearn.model_selection import train_test_split

In [62]:
import pandas as pd

In [63]:
df=pd.read_csv("weight_gain_data.csv")
X=df["calories"]
y=df["weight_gained_kg"]

In [64]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [65]:
model.fit(X_train,y_train)

Epoch 100 | Loss 0.0126 | w: 0.0814 | b: 0.0813
Epoch 200 | Loss 0.0123 | w: 0.0922 | b: 0.0920
Epoch 300 | Loss 0.0123 | w: 0.0937 | b: 0.0935
Epoch 400 | Loss 0.0123 | w: 0.0939 | b: 0.0937
Epoch 500 | Loss 0.0123 | w: 0.0939 | b: 0.0937
Epoch 600 | Loss 0.0123 | w: 0.0939 | b: 0.0937
Epoch 700 | Loss 0.0123 | w: 0.0939 | b: 0.0937
Epoch 800 | Loss 0.0123 | w: 0.0939 | b: 0.0937
Epoch 900 | Loss 0.0123 | w: 0.0939 | b: 0.0937
Epoch 1000 | Loss 0.0123 | w: 0.0939 | b: 0.0937


In [66]:
print(model.predict(X_test))

[ 0.09771675  0.1001901   0.03340954  0.08139261 -0.03868874  0.17822443
  0.15781926  0.24364465  0.00051392  0.16771268  0.10810484  0.15992161
  0.18947819 -0.04549047  0.14198979 -0.00702981  0.19145688  0.23486424
  0.19281722 -0.06230927  0.03996393  0.21371707  0.12578932  0.16350797
  0.25625876  0.23597725  0.18923086 -0.06725598 -0.02508529  0.0963564
  0.09610907  0.03093618 -0.08296178  0.03847991  0.23746127  0.13494073
 -0.01754156 -0.07541805  0.09784042  0.20258697]


In [67]:
from sklearn.metrics import mean_squared_error

In [68]:
mean_squared_error(model.predict(X_test),y_test)

0.022064042280221442

In [69]:
model.evaluate(X_test,y_test)

R^2 : 0.4228 (1.0) = perfect
MAE : 0.1112 kg (avg absolute error)
MSE : 0.0221 kg^2 (penalises big errors)
RMSE : 0.1485 kg (error in original units)
